
# FIQA / BEIR RAG Retriever Comparison

This notebook compares multiple **RAG-style retriever stacks** on the **BEIR FIQA** dataset using classic IR metrics:

- `avg_precision@5`, `avg_recall@5`, `avg_ndcg@5`
- `avg_precision@10`, `avg_recall@10`, `avg_ndcg@10`

We evaluate:

1. **BEIR-native retrievers**
   - BM25 (lexical)
   - Dense SBERT
   - Hybrid (lexical + dense)
2. **Haystack** BM25 retriever
3. **LangChain** FAISS dense retriever
4. **LlamaIndex** VectorStoreIndex retriever

All are normalized into a **BEIR-style results format** and evaluated via `EvaluateRetrieval` so that metrics are directly comparable.



## 0. Install dependencies

Uncomment and run this cell if you don't already have the libraries installed.


In [ ]:

# !pip install beir sentence-transformers pyserini
# !pip install farm-haystack
# !pip install "llama-index>=0.10.0"
# !pip install "langchain>=0.2.0" "langchain-community" datasets faiss-cpu


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.8/178.8 MB 31.6 MB/s  0:00:050m eta 0:00:010:01:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of pyserini to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.5/178.5 MB 36.8 MB/s  0:00:040m eta 0:00:010:01:02
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.5/178.5 MB 32.6 MB/s  0:00:050m eta 0:00:010:01:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.6/194.6 MB 26.9 MB/s  0:00:070m eta 0:00:010:01:01
  Installing build dependencies ... done
  Get


## 1. Common BEIR setup and metric helpers


In [7]:

import os
from typing import Dict, List
from pathlib import Path

from beir import util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval

# -----------------------------
# 1. Load FIQA dataset
# -----------------------------
def load_fiqa(split: str = "test"):
    # Use the full URL for FIQA dataset
    url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip"
    data_dir = Path("./datasets")
    data_dir.mkdir(exist_ok=True)
    
    # Download and unzip
    data_path = util.download_and_unzip(url, str(data_dir))
    
    # Find the actual data directory (might be nested)
    data_path = Path(data_path)
    if (data_path / "fiqa").exists():
        data_path = data_path / "fiqa"
    elif not (data_path / "corpus.jsonl").exists():
        # Search for corpus.jsonl
        corpus_files = list(data_path.rglob("corpus.jsonl"))
        if corpus_files:
            data_path = corpus_files[0].parent
    
    # Load the dataset
    corpus, queries, qrels = GenericDataLoader(str(data_path)).load(split=split)
    return corpus, queries, qrels

K_VALUES = [5, 10]


# -----------------------------
# 2. Metric helpers
# -----------------------------
def extract_metrics_dict(name, ndcg, map_, recall, prec, k_values=K_VALUES):
    """
    Convert BEIR metrics dicts into a flat dict with keys like:
        avg_precision@5, avg_recall@5, avg_ndcg@5, ...
    """
    out = {"pipeline": name}
    for k in k_values:
        out[f"avg_precision@{k}"] = prec[f"P@{k}"]
        out[f"avg_recall@{k}"]    = recall[f"R@{k}"]
        out[f"avg_ndcg@{k}"]      = ndcg[f"NDCG@{k}"]
        out[f"avg_map@{k}"]       = map_[f"MAP@{k}"]
    return out


def print_metrics_table(all_results: List[Dict], k_values=K_VALUES):
    """
    Pretty-print a comparison table across pipelines.
    """
    if not all_results:
        print("No results to display.")
        return

    header = ["pipeline"]
    for k in k_values:
        header += [f"P@{k}", f"R@{k}", f"nDCG@{k}"]
    print("\t".join(header))
    for res in all_results:
        row = [res["pipeline"]]
        for k in k_values:
            row += [
                f"{res.get(f'avg_precision@{k}', 0.0):.4f}",
                f"{res.get(f'avg_recall@{k}', 0.0):.4f}",
                f"{res.get(f'avg_ndcg@{k}', 0.0):.4f}",
            ]
        print("\t".join(row))



## 2. BEIR-native pipelines (BM25 / dense / hybrid)


In [ ]:

from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES
from beir.retrieval.models import SentenceBERT

def eval_beir_pipelines(corpus, queries, qrels, k_values=K_VALUES):
    metrics = []
    bm25_results = None
    dense_results = None

    # 1) BM25 lexical - Try Elasticsearch first, fallback to rank-bm25
    try:
        from beir.retrieval.search.lexical.elastic_search import ElasticSearch
        
        index_dir = "./indexes/fiqa-bm25"
        hostname = "localhost"
        initialize = True
        
        bm25_search = ElasticSearch(index_dir=index_dir, hostname=hostname, initialize=initialize)
        bm25_retriever = EvaluateRetrieval(bm25_search)
        
        bm25_results = bm25_retriever.retrieve(corpus, queries)
        bm25_ndcg, bm25_map, bm25_recall, bm25_prec = bm25_retriever.evaluate(
            qrels, bm25_results, k_values
        )
        metrics.append(
            extract_metrics_dict("beir_bm25_elasticsearch", bm25_ndcg, bm25_map, bm25_recall, bm25_prec, k_values)
        )
        print("✅ BM25 using Elasticsearch")
    except Exception as e1:
        print(f"⚠️ Elasticsearch BM25 failed: {e1}")
        # Fallback: Use rank-bm25 (pure Python, no Elasticsearch needed)
        try:
            from rank_bm25 import BM25Okapi
            import nltk
            from nltk.tokenize import word_tokenize
            
            # Download NLTK data if needed
            try:
                nltk.data.find('tokenizers/punkt')
            except LookupError:
                nltk.download('punkt', quiet=True)
            
            # Build BM25 index
            tokenized_corpus = []
            doc_ids = []
            for doc_id, doc in corpus.items():
                text = (doc.get("title", "") + " " + doc.get("text", "")).strip()
                tokens = word_tokenize(text.lower())
                tokenized_corpus.append(tokens)
                doc_ids.append(doc_id)
            
            bm25 = BM25Okapi(tokenized_corpus)
            
            # Retrieve for each query
            results = {}
            for qid, query_text in queries.items():
                query_tokens = word_tokenize(query_text.lower())
                scores = bm25.get_scores(query_tokens)
                # Get top-k documents
                top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:max(k_values)]
                results[qid] = {doc_ids[i]: float(scores[i]) for i in top_indices}
            
            # Evaluate
            dummy_eval = EvaluateRetrieval(None)
            bm25_ndcg, bm25_map, bm25_recall, bm25_prec = dummy_eval.evaluate(
                qrels, results, k_values
            )
            metrics.append(
                extract_metrics_dict("beir_bm25_rankbm25", bm25_ndcg, bm25_map, bm25_recall, bm25_prec, k_values)
            )
            bm25_results = results
            print("✅ BM25 using rank-bm25 (pure Python)")
        except Exception as e2:
            print(f"⚠️ rank-bm25 also failed: {e2}")
            print("   Skipping BM25. Install: pip install rank-bm25 nltk")

    # 2) Dense SBERT
    try:
        sbert_model = SentenceBERT("sentence-transformers/msmarco-distilbert-base-tas-b")
        dense_search = DRES(sbert_model, batch_size=128)
        dense_retriever = EvaluateRetrieval(dense_search, score_function="cos_sim")

        dense_results = dense_retriever.retrieve(corpus, queries)
        dense_ndcg, dense_map, dense_recall, dense_prec = dense_retriever.evaluate(
            qrels, dense_results, k_values
        )
        metrics.append(
            extract_metrics_dict("beir_dense_sbert", dense_ndcg, dense_map, dense_recall, dense_prec, k_values)
        )
        print("✅ Dense SBERT pipeline completed")
    except Exception as e:
        print(f"⚠️ Dense SBERT pipeline failed: {e}")

    # 3) Hybrid (lexical + dense) - only if both succeeded
    if bm25_results is not None and dense_results is not None:
        try:
            hybrid_retriever = EvaluateRetrieval(dense_search, score_function="cos_sim")
            hybrid_results = hybrid_retriever.hybrid(
                corpus, queries, bm25_results, dense_results, alpha=0.5
            )
            hybrid_ndcg, hybrid_map, hybrid_recall, hybrid_prec = hybrid_retriever.evaluate(
                qrels, hybrid_results, k_values
            )
            metrics.append(
                extract_metrics_dict("beir_hybrid_0.5", hybrid_ndcg, hybrid_map, hybrid_recall, hybrid_prec, k_values)
            )
            print("✅ Hybrid pipeline completed")
        except Exception as e:
            print(f"⚠️ Hybrid pipeline failed: {e}")
    
    return metrics




## 3. Haystack RAG retriever → BEIR metrics

We:
1. Load FIQA into a Haystack `DocumentStore`.
2. Use BM25Retriever.
3. Convert results into BEIR-style `results[qid] = {doc_id: score}`.
4. Use BEIR's `EvaluateRetrieval` just for metrics.


In [10]:

try:
    # Haystack v2 API
    from haystack import Document
    from haystack.document_stores import InMemoryDocumentStore
    from haystack.components.retrievers import BM25Retriever

    def build_haystack_store_from_corpus(corpus):
        """
        corpus: BEIR corpus dict {doc_id: {"title": ..., "text": ...}}
        """
        docs = []
        for doc_id, doc in corpus.items():
            text = (doc.get("title", "") + " " + doc.get("text", "")).strip()
            # Haystack v2 uses Document objects
            docs.append(Document(
                content=text,
                meta={"doc_id": doc_id}
            ))
        store = InMemoryDocumentStore()
        store.write_documents(docs)
        return store

    def haystack_results_to_beir_format(corpus, queries, retriever, top_k=10):
        """
        Returns BEIR-style results: results[qid] = {doc_id: score}
        """
        results = {}
        for qid, qtext in queries.items():
            # Haystack v2: retriever.run() returns a dict with 'documents' key
            output = retriever.run(query=qtext, top_k=top_k)
            hits = output.get("documents", [])
            scored_docs = {}
            for rank, d in enumerate(hits):
                doc_id = d.meta.get("doc_id", "")
                score = getattr(d, "score", None)
                if score is None:
                    score = 1.0 / (rank + 1)
                scored_docs[doc_id] = float(score)
            results[qid] = scored_docs
        return results

    def eval_haystack_pipeline(corpus, queries, qrels, k_values=K_VALUES):
        metrics = []
        store = build_haystack_store_from_corpus(corpus)
        # Haystack v2: BM25Retriever takes store as parameter
        haystack_bm25 = BM25Retriever(document_store=store)

        hs_results = haystack_results_to_beir_format(
            corpus, queries, haystack_bm25, top_k=max(k_values)
        )

        dummy_eval = EvaluateRetrieval(None)
        hs_ndcg, hs_map, hs_recall, hs_prec = dummy_eval.evaluate(
            qrels, hs_results, k_values
        )
        metrics.append(
            extract_metrics_dict("haystack_bm25", hs_ndcg, hs_map, hs_recall, hs_prec, k_values)
        )
        return metrics

except ImportError as e:
    def eval_haystack_pipeline(corpus, queries, qrels, k_values=K_VALUES):
        print(f"Haystack not installed or import error: {e}. Skipping Haystack pipeline.")
        return []



## 4. LangChain FAISS retriever → BEIR metrics

We:
1. Build a FAISS VectorStore from FIQA.
2. Use `as_retriever()` to get relevant documents.
3. Convert outputs to BEIR-style results.


In [11]:

try:
    from langchain_community.vectorstores import FAISS
    from langchain_community.embeddings import HuggingFaceEmbeddings

    def build_langchain_faiss_from_corpus(corpus):
        texts = []
        metadatas = []
        for doc_id, doc in corpus.items():
            text = (doc.get("title", "") + " " + doc.get("text", "")).strip()
            texts.append(text)
            metadatas.append({"doc_id": doc_id})
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/msmarco-distilbert-base-tas-b"
        )
        vs = FAISS.from_texts(texts, embedding=embeddings, metadatas=metadatas)
        return vs

    def langchain_results_to_beir_format(corpus, queries, retriever, top_k=10):
        results = {}
        for qid, qtext in queries.items():
            docs = retriever.get_relevant_documents(qtext)[:top_k]
            scored_docs = {}
            for rank, d in enumerate(docs):
                doc_id = d.metadata["doc_id"]
                score = d.metadata.get("score", 1.0/(rank+1))
                scored_docs[doc_id] = float(score)
            results[qid] = scored_docs
        return results

    def eval_langchain_pipeline(corpus, queries, qrels, k_values=K_VALUES):
        metrics = []
        vs = build_langchain_faiss_from_corpus(corpus)
        retriever = vs.as_retriever(search_kwargs={"k": max(k_values)})

        lc_results = langchain_results_to_beir_format(
            corpus, queries, retriever, top_k=max(k_values)
        )
        dummy_eval = EvaluateRetrieval(None)
        lc_ndcg, lc_map, lc_recall, lc_prec = dummy_eval.evaluate(
            qrels, lc_results, k_values
        )
        metrics.append(
            extract_metrics_dict("langchain_faiss", lc_ndcg, lc_map, lc_recall, lc_prec, k_values)
        )
        return metrics

except ImportError:
    def eval_langchain_pipeline(corpus, queries, qrels, k_values=K_VALUES):
        print("LangChain / FAISS not installed, skipping LangChain pipeline.")
        return []



## 5. LlamaIndex VectorStoreIndex retriever → BEIR metrics

We:
1. Build a `VectorStoreIndex` from FIQA.
2. Use `as_retriever()` to retrieve nodes.
3. Convert to BEIR-style results.


In [12]:

try:
    from llama_index.core import Document as LIDocument
    from llama_index.core import VectorStoreIndex

    def build_llamaindex_index_from_corpus(corpus):
        docs = []
        for doc_id, doc in corpus.items():
            text = (doc.get("title", "") + " " + doc.get("text", "")).strip()
            docs.append(
                LIDocument(
                    text=text,
                    metadata={"doc_id": doc_id}
                )
            )
        index = VectorStoreIndex.from_documents(docs)
        return index

    def llamaindex_results_to_beir_format(corpus, queries, retriever, top_k=10):
        results = {}
        for qid, qtext in queries.items():
            nodes = retriever.retrieve(qtext)
            nodes = nodes[:top_k]
            scored_docs = {}
            for rank, n in enumerate(nodes):
                doc_id = n.metadata["doc_id"]
                score = getattr(n, "score", None)
                if score is None:
                    score = getattr(n, "similarity", 1.0/(rank+1))
                scored_docs[doc_id] = float(score)
            results[qid] = scored_docs
        return results

    def eval_llamaindex_pipeline(corpus, queries, qrels, k_values=K_VALUES):
        metrics = []
        index = build_llamaindex_index_from_corpus(corpus)
        retriever = index.as_retriever(similarity_top_k=max(k_values))

        li_results = llamaindex_results_to_beir_format(
            corpus, queries, retriever, top_k=max(k_values)
        )
        dummy_eval = EvaluateRetrieval(None)
        li_ndcg, li_map, li_recall, li_prec = dummy_eval.evaluate(
            qrels, li_results, k_values
        )
        metrics.append(
            extract_metrics_dict("llamaindex_vector", li_ndcg, li_map, li_recall, li_prec, k_values)
        )
        return metrics

except ImportError:
    def eval_llamaindex_pipeline(corpus, queries, qrels, k_values=K_VALUES):
        print("LlamaIndex not installed, skipping LlamaIndex pipeline.")
        return []



## 6. Run all pipelines and compare


In [13]:

# This may take a while the first time (downloads FIQA, builds indexes, computes embeddings).

corpus, queries, qrels = load_fiqa(split="test")

all_metrics = []

# 1) BEIR-native
print("Running BEIR-native pipelines...")
all_metrics.extend(eval_beir_pipelines(corpus, queries, qrels))

# 2) Haystack
print("Running Haystack pipeline...")
all_metrics.extend(eval_haystack_pipeline(corpus, queries, qrels))

# 3) LangChain
print("Running LangChain pipeline...")
all_metrics.extend(eval_langchain_pipeline(corpus, queries, qrels))

# 4) LlamaIndex
print("Running LlamaIndex pipeline...")
all_metrics.extend(eval_llamaindex_pipeline(corpus, queries, qrels))

print("\n=== Comparison Table ===")
print_metrics_table(all_metrics, k_values=K_VALUES)


100%|██████████| 57638/57638 [00:00<00:00, 265834.87it/s]


Running BEIR-native pipelines...
⚠️ Elasticsearch BM25 failed: No module named 'beir.retrieval.search.lexical.elasticsearch_search'
⚠️ rank-bm25 also failed: 'R@5'
   Skipping BM25. Install: pip install rank-bm25 nltk


Batches:  26%|██▋       | 103/391 [07:46<21:44,  4.53s/it] 


KeyboardInterrupt: 